In [1]:
#clean url

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from tqdm import tqdm

/Users/ankit/Projects/learn_llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OLD_DB = "vector_db/ircc"
NEW_DB = "vector_db/ircc_cleaned"

embedding = OllamaEmbeddings(model="qwen3-embedding:0.6b")

# Load existing DB
old_store = Chroma(
    persist_directory=OLD_DB,
    embedding_function=embedding
)

# Create new DB
new_store = Chroma(
    persist_directory=NEW_DB,
    embedding_function=embedding
)

print("Total docs in old DB:", old_store._collection.count())

/var/folders/ng/dp6yvncx7pl1qbny9711g9hr0000gn/T/ipykernel_20800/2504068223.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  old_store = Chroma(


Total docs in old DB: 71873


In [3]:
import itertools

In [4]:
BATCH_SIZE = 1000

documents = []
metadatas = []
embeddings = []
ids = []

for offset in tqdm(range(0, old_store._collection.count(), BATCH_SIZE)):

    batch = old_store._collection.get(
        limit=BATCH_SIZE,
        offset=offset,
        include=["documents", "metadatas", "embeddings"]
    )
    documents.append(batch["documents"])
    metadatas.append(batch["metadatas"])
    embeddings.append(batch["embeddings"])
    ids.append(batch["ids"])

100%|███████████████████████████████████████████████████████████████████████████████████| 72/72 [00:10<00:00,  7.18it/s]


In [5]:
documents = list(itertools.chain.from_iterable(documents))
metadatas = list(itertools.chain.from_iterable(metadatas))
embeddings = list(itertools.chain.from_iterable(embeddings))
ids = list(itertools.chain.from_iterable(ids))

In [6]:
seen_docs = set()

clean_docs = []
clean_metas = []
clean_embeds = []
clean_ids = []

In [7]:
for doc, meta, embed, id_ in tqdm(zip(documents, metadatas, embeddings, ids), total=len(ids)):

    if doc in seen_docs:
        continue

    seen_docs.add(doc)

    clean_docs.append(doc)
    clean_metas.append(meta)
    clean_embeds.append(embed)
    clean_ids.append(id_)

100%|█████████████████████████████████████████████████████████████████████████| 71873/71873 [00:00<00:00, 577237.73it/s]


In [8]:
len(clean_docs)

9283

In [12]:
print(clean_docs[100])

Title: Guide 5552 - Applying to Change Conditions or Extend Your Stay in Canada - Student - online application
Description: Guide 5552 - Applying to Change Conditions or Extend Your Stay in Canada - Student - online application
Keywords: Applications; Temporary resident visas; Study permits

Section: Services and Information
Content:
For more information on the programs offered by IRCC, visit our website .


In [51]:
# for u in metadatas[:1000]:
#     if "#".lower() in u['url'].lower(): 
#        print(u['url']) 

In [13]:
len(clean_docs)

9283

In [14]:
print("After cleaning:", len(clean_docs))

for i in tqdm(range(0, len(clean_docs), BATCH_SIZE)):

    # Add to new collection WITHOUT re-embedding
    new_store._collection.add(
        documents=clean_docs[i:i + BATCH_SIZE],
        metadatas=clean_metas[i:i + BATCH_SIZE],
        embeddings=clean_embeds[i:i + BATCH_SIZE],
        ids=clean_ids[i:i + BATCH_SIZE]
    )
    
print("Clean DB created.")

After cleaning: 9283


 20%|████████████████▊                                                                   | 2/10 [00:00<00:03,  2.38it/s]

Clean DB created.
Clean DB created.


 30%|█████████████████████████▏                                                          | 3/10 [00:01<00:03,  2.22it/s]

Clean DB created.
Clean DB created.


...

 80%|███████████████████████████████████████████████████████████████████▏                | 8/10 [00:04<00:01,  1.44it/s]

Clean DB created.
Clean DB created.


 90%|███████████████████████████████████████████████████████████████████████████▌        | 9/10 [00:05<00:00,  1.40it/s]

Clean DB created.
Clean DB created.


100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:05<00:00,  1.72it/s]


In [60]:
clean_metas[0]['date']

'2025-09-11'

In [66]:

date_string = "2025-01-01"

date_object = datetime.strptime(date_string, "%Y-%m-%d")
date_object.timestamp()

1735707600.0